# 12주차 — Estimation 팔레트: 필터로 복구

```
[Sensors] → [Estimation] → [Navigation] → [Control] → [Allocation] → [Plant]
              ▲ 이번 주
```

목표: 상보필터(Mahony)의 원리 — 자이로 적분(고주파) + 가속도계 보정(저주파)
+ **바이어스 추정**. 11주차의 흔들림 중 '바이어스' 성분을 걷어낸다.

## 이론 요약
- 자이로만: 적분 드리프트 (바이어스가 각도로 쌓인다)
- 가속도계만: 기동/진동에 떨린다 (비력 ≠ 중력)
- 융합: ω_corr = gyro − b̂ + Kp·e,  b̂˙ = −Ki·e,  e = 측정중력 × 추정중력
- **정직한 한계 (중요)**: 추력만 받는 기체는 기울면 수평가속하며 비력이 항상
  body−z — 가속 중 tilt 관측 불가. CF 자세추정을 제어 자세 피드백에 그대로
  쓰면 발산한다! 그래서 이번 주는 **바이어스 보정 각속도(omega_est)만** 쓴다.
  자세까지 추정으로 날려면 GPS 속도 결합(EKF2)이 필요 — MATLAB 라인 T1 수업
  의 예고편이다 (px4_controller/nav/EKF2_gravity_notes 참조).

In [ ]:
# 뷰어 열기 — 뼈대 모델을 브라우저에서 편집한다.
# 완성하면 뷰어의 '저장' → week12_model.json 이 다운로드된다.
# 그 파일을 이 폴더의 같은 이름 파일에 **덮어쓰면 제출 완료** (채점 셀로 확인).
import sys, webbrowser; sys.path.insert(0, '../..')
from loopsym import load_json, render_html
p = render_html(load_json('week12_model.json'), 'week12_viewer.html',
                title='12주차', t_end=8.0, dt=0.004,
                save_name='week12_model.json')
webbrowser.open('file://' + p)

## 실습 — CF 삽입
① ComplementaryFilter 블록 삽입: IMU 의 gyro+accel 6줄 → CF 입력.
② rate_ctrl 의 각속도 피드백을 IMU gyro → **CF 의 p/q/r (omega_est)** 로 재배선.
③ 실행 — 11주차 대비 바이어스로 인한 자세 오프셋이 사라지는 것을 확인.

In [ ]:
# 노트북에서 바로 실행해 보기 (뷰어 없이) — 저장한 모델을 로드해 시뮬
import sys; sys.path.insert(0, '../..')
from loopsym import load_json, Simulator
bd = load_json('week12_model.json')
sim = Simulator(bd); sim.run(8.0, 0.004)
import matplotlib.pyplot as plt
for b in sim.bd.blocks:
    if type(b).__name__ == 'Scope':
        plt.figure(figsize=(8,2.6)); plt.title(b.name); plt.grid(True)
        for i, lab in enumerate(b.labels): plt.plot(b.t, b.data[i], label=lab)
        plt.legend()

## 과제
1. week12_model.json 완성 (채점기: CF 구조 + 미션 + 'CF→자세 직결 금지')
2. 트레이드오프 리포트: CF 의 ki 를 0.05→0.5 로 키우면 바이어스 수렴은
   빨라지지만 무엇이 나빠지는가? (힌트: 진동 전달)

In [ ]:
# ✅ 채점 — 전부 초록색이 되면 이번 주 완료!
!python -m pytest test_week12.py -q